# 019 Paper Tables Overview

One cell per table, arranged in the order they appear in the tex source.
All data sources are read from existing experiment result CSVs; nothing is recomputed.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

BR_RESULTS = Path('./results')
DE_RESULTS = Path('../Germany/results')
STATIC_DIR = BR_RESULTS / 'static_allocation'
EXP0_DIR   = BR_RESULTS / 'exp0_kfold_prior'
EXP1_DIR   = BR_RESULTS / 'exp1_main_table'
EXP2_DIR   = BR_RESULTS / 'exp2_significance'
EXP6_DIR   = BR_RESULTS / 'exp6_heterogeneity'
H1H3_DIR   = BR_RESULTS / 'exp_h1h3_mechanism_isolation'
H2_DIR     = BR_RESULTS / 'exp_h2_embedding_probing'

SEEDS = [42, 123, 456]
LOCS = [
    'London','TLH2','TLH3','TLJ1','TLF1','TLF2',
    'TLC1','TLC2','TLD6','TLG1','TLG2','TLE4',
    'TLH1','TLE3','TLD3','TLD4',
]

print('All paths OK:', all(p.exists() for p in [STATIC_DIR, EXP1_DIR, EXP2_DIR, EXP6_DIR, H1H3_DIR, H2_DIR, DE_RESULTS]))

---
## Table 1 · `tab:main` — Aggregate Performance (15 methods)

In [ ]:
tab1 = pd.read_csv(EXP1_DIR / 'table1_main_results.csv')
display(tab1[['label', 'type', 'rmse', 'mae', 'corr']])

---
## Table 2 · `tab:significance` — Pairwise Statistical Tests

In [ ]:
sig = pd.read_csv(EXP2_DIR / 'significance_tests.csv')

# The paper reports only the 9 RMSE-dimension comparisons
sig_rmse = sig[sig['metric'] == 'rmse'][['comparison', 'mean_diff', 'holm_p', 'significant_005']].copy()
sig_rmse.columns = ['Comparison', 'ΔRMSE', 'Holm p', 'Sig.']
sig_rmse = sig_rmse.reset_index(drop=True)
sig_rmse.index += 1
sig_rmse.index.name = '#'
display(sig_rmse)

---
## Table 3 · `tab:decoupling` — RMSE–Corr Decoupling

In [ ]:
# Extract the RMSE and Corr dimension comparisons from significance_tests.csv
sig = pd.read_csv(EXP2_DIR / 'significance_tests.csv')

decoupling_pairs = [
    ('G2b vs G1', 'GNNpostN vs GNN'),
    ('G2a vs G1', 'GNNpostP vs GNN'),
    ('G3 vs G2a', 'GNNpostNP vs GNNpostP'),
    ('S4 vs S3b', 'GPMpostNP vs GPMpostN'),
]

rows = []
for label, comp_key in decoupling_pairs:
    rmse_row = sig[(sig['metric'] == 'rmse') & (sig['comparison'] == comp_key)].iloc[0]
    corr_row = sig[(sig['metric'] == 'corr') & (sig['comparison'] == comp_key)].iloc[0]
    rows.append({
        'Comparison': label,
        'ΔRMSE': f"{rmse_row['mean_diff']:+.2f}",
        'RMSE Sig.': '✓' if rmse_row['significant_005'] else '—',
        'RMSE Holm p': f"{rmse_row['holm_p']:.4g}",
        'ΔCorr': f"{corr_row['mean_diff']:+.3f}",
        'Corr Sig.': '✓' if corr_row['significant_005'] else '—',
        'Corr Holm p': f"{corr_row['holm_p']:.4g}",
    })

tab_decoupling = pd.DataFrame(rows)
display(tab_decoupling)

---
## Table 4 · `tab:asymmetry` — Synergy–Antagonism Asymmetry (Independent Marginals)

In [ ]:
# Static methods: independent marginal effect from static CSV
rmse_static = pd.read_csv(STATIC_DIR / 'all_regions_rmse.csv', index_col=0)

def static_delta(base_m, added_m):
    b = rmse_static.loc[base_m, LOCS].astype(float).values
    a = rmse_static.loc[added_m, LOCS].astype(float).values
    d = (a - b).mean()
    pct = d / b.mean() * 100
    return d, pct

# GNN methods: independent marginal effect from kfold CSVs
def gnn_delta(base_method, added_method, config='baseline'):
    deltas = []
    for seed in SEEDS:
        csv = EXP0_DIR / f'seed_{seed}' / config / 'kfold_test_rmse.csv'
        df = pd.read_csv(csv, index_col=0)
        b = df.loc[base_method].values.astype(float).mean()
        a = df.loc[added_method].values.astype(float).mean()
        deltas.append(a - b)
    d = np.mean(deltas)
    base_avg = np.mean([pd.read_csv(EXP0_DIR / f'seed_{s}' / config / 'kfold_test_rmse.csv', index_col=0)
                        .loc[base_method].values.astype(float).mean() for s in SEEDS])
    return d, d / base_avg * 100

# GNN prior: voronoi_GNN under each prior config vs voronoi_GNN under baseline
def gnn_prior_delta(prior_config):
    deltas = []
    for seed in SEEDS:
        base_csv = EXP0_DIR / f'seed_{seed}' / 'baseline' / 'kfold_test_rmse.csv'
        prior_csv = EXP0_DIR / f'seed_{seed}' / prior_config / 'kfold_test_rmse.csv'
        b = pd.read_csv(base_csv, index_col=0).loc['voronoi_GNN'].values.astype(float).mean()
        a = pd.read_csv(prior_csv, index_col=0).loc['voronoi_GNN'].values.astype(float).mean()
        deltas.append(a - b)
    d = np.mean(deltas)
    base_avg = np.mean([pd.read_csv(EXP0_DIR / f'seed_{s}' / 'baseline' / 'kfold_test_rmse.csv', index_col=0)
                        .loc['voronoi_GNN'].values.astype(float).mean() for s in SEEDS])
    return d, d / base_avg * 100

def fmt(d, pct): return f'{d:+.2f} ({pct:+.1f}%)'

# Build the table
uni_ntl, uni_ntl_p = static_delta('voronoi', 'voronoi_ntl')
uni_prox, uni_prox_p = static_delta('voronoi', 'voronoi_prox2')
uni_np, uni_np_p = static_delta('voronoi', 'voronoi_prox2_ntl')

gpm_ntl, gpm_ntl_p = static_delta('voronoi_gpm', 'voronoi_ntl_gpm')
gpm_prox, gpm_prox_p = static_delta('voronoi_gpm', 'voronoi_prox2_gpm')
gpm_np, gpm_np_p = static_delta('voronoi_gpm', 'voronoi_prox2_ntl_gpm')

gnn_ntl, gnn_ntl_p = gnn_delta('voronoi_GNN', 'voronoi_ntl_GNN')
gnn_prox, gnn_prox_p = gnn_delta('voronoi_GNN', 'voronoi_prox_GNN')
gnn_np, gnn_np_p = gnn_delta('voronoi_GNN', 'voronoi_ntl_prox_GNN')

prior_ntl, prior_ntl_p = gnn_prior_delta('ntl')
prior_prox, prior_prox_p = gnn_prior_delta('proximity')
prior_np, prior_np_p = gnn_prior_delta('ntl_prox')

tab_asym = pd.DataFrame([
    {'Component': 'NTL',      'Uniform base': fmt(uni_ntl, uni_ntl_p), 'GPM base': fmt(gpm_ntl, gpm_ntl_p), 'GNN post-corr.': fmt(gnn_ntl, gnn_ntl_p), 'GNN prior loss': fmt(prior_ntl, prior_ntl_p)},
    {'Component': 'Proximity','Uniform base': fmt(uni_prox, uni_prox_p),'GPM base': fmt(gpm_prox, gpm_prox_p),'GNN post-corr.': fmt(gnn_prox, gnn_prox_p),'GNN prior loss': fmt(prior_prox, prior_prox_p)},
    {'Component': 'NTL+Prox', 'Uniform base': fmt(uni_np, uni_np_p),  'GPM base': fmt(gpm_np, gpm_np_p),  'GNN post-corr.': fmt(gnn_np, gnn_np_p),  'GNN prior loss': fmt(prior_np, prior_np_p)},
]).set_index('Component')
display(tab_asym)

# Additivity check
print('\n=== Additivity check ===')
for name, d_n, d_p, d_np in [('Uniform', uni_ntl, uni_prox, uni_np), ('GPM', gpm_ntl, gpm_prox, gpm_np), ('GNN post', gnn_ntl, gnn_prox, gnn_np)]:
    s = d_n + d_p
    ratio = d_np / s if s != 0 else float('inf')
    print(f'  {name:8s}: NTL({d_n:+.2f}) + Prox({d_p:+.2f}) = {s:+.2f}  vs  combined = {d_np:+.2f}  ratio = {ratio:.2f}')

---
## Table 5 · `tab:heterogeneity` — Regional Heterogeneity (Density × Diversity)

In [ ]:
tab_het = pd.read_csv(EXP6_DIR / 'table3_cross_group.csv')
col_map = {
    'density_group':                    'Density',
    'entropy_group':                    'Diversity',
    'Uni (voronoi uniform)':            'Uni',
    'GPMpostNP (static-optimal)':              'GPMpostNP',
    'GNN (baseline)':                   'GNN',
    'GNNpostP (Prox post-hoc)':           'GNNpostP',
    'GNNpostNP (NTL+Prox post-hoc)':      'GNNpostNP',
    'GNNpriorNP (NTL+Prox prior)':       'GNNpriorNP',
}
tab5 = tab_het.rename(columns=col_map)[[c for c in col_map.values() if c in tab_het.rename(columns=col_map).columns]]
display(tab5)

---
## Table 6 · `tab:mechanism_isolation` — Mechanism Isolation Experiments

In [ ]:
raw = pd.read_csv(H1H3_DIR / 'mechanism_isolation_raw.csv')

# Consistent with Tab.3's convention: first average over seeds within each location, then compute the standard deviation across 16 locations with ddof=0
non_noise = raw[~raw['experiment'].str.contains('random_noise')]

# Step 1: average over seeds within the same (experiment, location)
loc_avg = non_noise.groupby(['experiment', 'location'])[['rmse', 'mae', 'corr']].mean()

# Step 2: compute the mean and population standard deviation across the 16 locations (ddof=0)
summary = loc_avg.groupby('experiment').agg(
    rmse_mean=('rmse', 'mean'), rmse_std=('rmse', lambda x: x.std(ddof=0)),
    mae_mean=('mae',  'mean'),  mae_std=('mae',  lambda x: x.std(ddof=0)),
    corr_mean=('corr','mean'),  corr_std=('corr',lambda x: x.std(ddof=0)),
)

# Random noise: average over reps first -> then average over seeds -> ddof=0 across 16 locations
noise = raw[raw['experiment'].str.contains('random_noise')].copy()
noise['rep'] = noise['experiment'].str.extract(r'rep(\d+)').astype(int)
noise_per_loc_seed = noise.groupby(['seed', 'location'])[['rmse', 'mae', 'corr']].mean()
noise_per_loc = noise_per_loc_seed.groupby('location').mean()
noise_row = pd.DataFrame([{
    'rmse_mean': noise_per_loc['rmse'].mean(),
    'rmse_std':  noise_per_loc['rmse'].std(ddof=0),
    'mae_mean':  noise_per_loc['mae'].mean(),
    'mae_std':   noise_per_loc['mae'].std(ddof=0),
    'corr_mean': noise_per_loc['corr'].mean(),
    'corr_std':  noise_per_loc['corr'].std(ddof=0),
}], index=['H1b_random_noise (10 reps avg)'])

tab6 = pd.concat([summary, noise_row])

baseline_rmse = tab6.loc['baseline_no_correction', 'rmse_mean']
tab6['ΔRMSE']  = tab6['rmse_mean'] - baseline_rmse
tab6['ΔRMSE%'] = tab6['ΔRMSE'] / baseline_rmse * 100

tab6_display = tab6[['rmse_mean', 'rmse_std', 'ΔRMSE', 'ΔRMSE%', 'mae_mean', 'mae_std', 'corr_mean', 'corr_std']].copy()
tab6_display = tab6_display.sort_values('rmse_mean')
tab6_display.index.name = 'Experiment'

# Rename the row index to the display names used in the paper (consistent with Tab. 7 / tex)
DISPLAY_NAMES = {
    'baseline_no_correction':          'GNN baseline (no correction)',
    'standard_multiplicative_NP':      'GNN + NTL×Prox (Mult.)',
    'standard_multiplicative_N':       'GNN + NTL only (Mult.)',
    'standard_multiplicative_P':       'GNN + Prox only (Mult.)',
    'H1a_no_renorm_NP':                'No-renorm NTL×Prox',
    'H1a_no_renorm_N':                 'No-renorm NTL only',
    'H1a_no_renorm_P':                 'No-renorm Prox only',
    'H1b_random_noise (10 reps avg)':  'Random noise (10 repeats avg.)',
    'H3_additive_NP':                  'Additive NTL×Prox',
    'H3_additive_N':                   'Additive NTL only',
    'H3_additive_P':                   'Additive Prox only',
}
tab6_display = tab6_display.rename(index=DISPLAY_NAMES)

display(tab6_display.round(4))

---
## Table 7 · `tab:embedding_probing` — Embedding Probing

In [ ]:
emb = pd.read_csv(H2_DIR / 'embedding_probing_summary.csv', header=[0, 1], index_col=0)
display(emb)

---
## Table 8 · `tab:tau` — Learned Temperature τ

In [ ]:
tau = pd.read_csv(H2_DIR / 'tau_values.csv')
tau_summary = tau.groupby('config')['tau'].agg(['mean', 'std', 'min', 'max'])
tau_summary.index.name = 'GNN config'
display(tau_summary)

---
## Table 9 · `tab:germany` — German Börde Key Results

In [ ]:
de = pd.read_csv(DE_RESULTS / 'comparison_table.csv')

# Key methods reported in the paper
key_methods = [
    'voronoi',            # Uni
    'voronoi_gpm',        # GPM
    'voronoi_prox2_ntl',  # UniNP
    'gemeinde_average',   # Oracle
]
key_gnn = [
    ('baseline', 'voronoi_GNN'),      # GNN
    ('ntl_prox', 'voronoi_GNN'),      # GNNpriorNP
]

rows = []
for m in key_methods:
    r = de[de['method'] == m]
    if len(r) > 0:
        r = r.iloc[0]
        rows.append({'Method': m, 'RMSE': r.get('rmse', r.get('rmse_val', '')), 'Corr': r.get('corr', r.get('corr_val', ''))})

for cfg, m in key_gnn:
    r = de[(de['method'] == m) & (de['type'].str.contains(cfg))]
    if len(r) == 0:
        r = de[de['method'].str.contains(m) & de['type'].str.contains(cfg)]
    if len(r) > 0:
        r = r.iloc[0]
        rows.append({'Method': f'{m} [{cfg}]', 'RMSE': r.get('rmse', ''), 'Corr': r.get('corr', '')})

tab_de = pd.DataFrame(rows)
display(tab_de)

---
## Table 10 · `tab:prior_boundary` — Prior-Loss Effect: Britain vs Germany

In [ ]:
# Britain: from kfold CSVs
br_deltas = {}
for prior_cfg, label in [('ntl', 'NTL only'), ('proximity', 'Prox only'), ('ntl_prox', 'NTL+Prox')]:
    deltas = []
    for seed in SEEDS:
        base = pd.read_csv(EXP0_DIR / f'seed_{seed}' / 'baseline' / 'kfold_test_rmse.csv', index_col=0)
        prior = pd.read_csv(EXP0_DIR / f'seed_{seed}' / prior_cfg / 'kfold_test_rmse.csv', index_col=0)
        b = base.loc['voronoi_GNN'].values.astype(float).mean()
        p = prior.loc['voronoi_GNN'].values.astype(float).mean()
        deltas.append(p - b)
    d = np.mean(deltas)
    base_avg = np.mean([pd.read_csv(EXP0_DIR / f'seed_{s}' / 'baseline' / 'kfold_test_rmse.csv', index_col=0)
                        .loc['voronoi_GNN'].values.astype(float).mean() for s in SEEDS])
    br_deltas[label] = (d, d / base_avg * 100)

# Germany: from model summary CSVs
de_deltas = {}
de_base_metrics = []
for seed in SEEDS:
    m = pd.read_csv(DE_RESULTS / f'models/baseline/seed_{seed}/metrics.csv')
    gnn_row = m[m['method'] == 'voronoi_GNN']
    if len(gnn_row) > 0:
        de_base_metrics.append(gnn_row.iloc[0]['rmse'])
de_base_avg = np.mean(de_base_metrics) if de_base_metrics else None

for prior_cfg, label in [('ntl', 'NTL only'), ('proximity', 'Prox only'), ('ntl_prox', 'NTL+Prox')]:
    prior_vals = []
    for seed in SEEDS:
        m = pd.read_csv(DE_RESULTS / f'models/{prior_cfg}/seed_{seed}/metrics.csv')
        gnn_row = m[m['method'] == 'voronoi_GNN']
        if len(gnn_row) > 0:
            prior_vals.append(gnn_row.iloc[0]['rmse'])
    if prior_vals and de_base_avg:
        d = np.mean(prior_vals) - de_base_avg
        de_deltas[label] = (d, d / de_base_avg * 100)

rows = []
for label in ['NTL only', 'Prox only', 'NTL+Prox']:
    br_d, br_p = br_deltas.get(label, (None, None))
    de_d, de_p = de_deltas.get(label, (None, None))
    rows.append({
        'Prior config': label,
        'Britain ΔRMSE': f'{br_d:+.2f} ({br_p:+.1f}%)' if br_d is not None else '—',
        'Germany ΔRMSE': f'{de_d:+.2f} ({de_p:+.1f}%)' if de_d is not None else '—',
    })
display(pd.DataFrame(rows))